# Import

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import make_scorer, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.losses import MeanSquaredError, mse
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping

from scikeras.wrappers import KerasRegressor

# 원본 데이터 전처리

In [5]:
# 3개년 데이터 로드 (인덱스 자동추가 제외)
origin_2021 = pd.read_csv('./train_subway21.csv', index_col=0)
origin_2022 = pd.read_csv('./train_subway22.csv', index_col=0)
origin_2023 = pd.read_csv('./train_subway23.csv', index_col=0)

In [6]:
# 칼럼명 변경
origin_2021 = origin_2021.rename(columns={
    'train_subway21.tm': 'time',
    'train_subway21.line': 'line',
    'train_subway21.station_number': 'station_number',
    'train_subway21.station_name': 'station_name',
    'train_subway21.direction': 'direction',
    'train_subway21.stn': 'AWS_num',
    'train_subway21.ta': 'temp',
    'train_subway21.wd': 'wind_direction',
    'train_subway21.ws': 'wind_speed',
    'train_subway21.rn_day': 'rn_day',
    'train_subway21.rn_hr1': 'rn_hr',
    'train_subway21.hm': 'humidity',
    'train_subway21.si': 'solar',
    'train_subway21.ta_chi': 'feel_temp',
    'train_subway21.congestion': 'congestion'
})

origin_2022 = origin_2022.rename(columns={
    'train_subway22.tm': 'time',
    'train_subway22.line': 'line',
    'train_subway22.station_number': 'station_number',
    'train_subway22.station_name': 'station_name',
    'train_subway22.direction': 'direction',
    'train_subway22.stn': 'AWS_num',
    'train_subway22.ta': 'temp',
    'train_subway22.wd': 'wind_direction',
    'train_subway22.ws': 'wind_speed',
    'train_subway22.rn_day': 'rn_day',
    'train_subway22.rn_hr1': 'rn_hr',
    'train_subway22.hm': 'humidity',
    'train_subway22.si': 'solar',
    'train_subway22.ta_chi': 'feel_temp',
    'train_subway22.congestion': 'congestion'
})

origin_2023 = origin_2023.rename(columns={
    'train_subway23.tm': 'time',
    'train_subway23.line': 'line',
    'train_subway23.station_number': 'station_number',
    'train_subway23.station_name': 'station_name',
    'train_subway23.direction': 'direction',
    'train_subway23.stn': 'AWS_num',
    'train_subway23.ta': 'temp',
    'train_subway23.wd': 'wind_direction',
    'train_subway23.ws': 'wind_speed',
    'train_subway23.rn_day': 'rn_day',
    'train_subway23.rn_hr1': 'rn_hr',
    'train_subway23.hm': 'humidity',
    'train_subway23.si': 'solar',
    'train_subway23.ta_chi': 'feel_temp',
    'train_subway23.congestion': 'congestion'
})

In [7]:
# origin dataframe 통합
origin_all = pd.concat([origin_2021, origin_2022, origin_2023], ignore_index=True)
origin_all

,time,line,station_number,station_name,direction,AWS_num,temp,wind_direction,wind_speed,rn_day,rn_hr,humidity,solar,feel_temp,congestion
0,2021010100,1,150,서울역,상선,419,-9.6,291.1,3.3,0.0,0.0,-99.0,-99.0,-12.6,0
1,2021010101,1,150,서울역,상선,419,-9.7,284.6,2.0,0.0,0.0,-99.0,-99.0,-9.8,0
2,2021010105,1,150,서울역,상선,419,-9.3,124.7,2.4,0.0,0.0,-99.0,-99.0,-10.3,1
3,2021010106,1,150,서울역,상선,419,-9.3,126.2,1.7,0.0,0.0,-99.0,-99.0,-10.1,2
4,2021010107,1,150,서울역,상선,419,-9.1,145.7,1.3,0.0,0.0,-99.0,-99.0,-9.7,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16369319,2023123119,8,2828,남위례,하선,572,0.6,0.0,0.0,7.0,0.0,83.1,-99.0,0.0,18
16369320,2023123120,8,2828,남위례,하선,572,0.0,354.7,0.0,7.0,0.0,84.7,-99.0,-0.6,17
16369321,2023123121,8,2828,남위례,하선,572,-0.6,0.0,0.0,7.0,0.0,85.1,-99.0,-1.1,21
16369322,2023123122,8,2828,남위례,하선,572,-0.8,0.0,0.0,7.0,0.0,85.6,-99.0,-1.3,18


In [8]:
# 결측값 → NaN
sentinels = [-99, -99.9, -999.0]
origin_all = origin_all.replace(sentinels, np.nan)

In [9]:
# 결측치 확인
print('결측치 처리 전 결측치 개수')
print(origin_all.isnull().sum())

결측치 처리 전 결측치 개수
time                    0
line                    0
station_number          0
station_name            0
direction               0
AWS_num                 0
temp               216468
wind_direction     230786
wind_speed         230786
rn_day             351574
rn_hr              360796
humidity           844594
solar             6064242
feel_temp             352
congestion              0
dtype: int64


In [10]:
# 결측치 발생 컬럼 그룹화
target_cols = ['temp', 'wind_direction', 'wind_speed', 'rn_day', 'rn_hr', 'humidity', 'solar', 'feel_temp']

# AWS_num과 시간 기준 정렬
origin_all = origin_all.sort_values(by=['AWS_num', 'time'])

# 그룹별 보간
origin_all[target_cols] = (
    origin_all.groupby('AWS_num')[target_cols]
    .transform(lambda grp: grp.bfill().ffill())
)

# 시간 기준 재정렬
origin_all = origin_all.sort_values(by='time').reset_index(drop=True)

In [11]:
# 결측치 재확인
print('결측치 처리 후 결측치 개수')
print(origin_all.isnull().sum())

결측치 처리 후 결측치 개수
time              0
line              0
station_number    0
station_name      0
direction         0
AWS_num           0
temp              0
wind_direction    0
wind_speed        0
rn_day            0
rn_hr             0
humidity          0
solar             0
feel_temp         0
congestion        0
dtype: int64


In [12]:
# 상/하행 레이블 정의
direction_map = {'상선': 0, '하선': 1, '내선': 2, '외선':3}

# 체감온도 구간 레이블 정의
bins = [-float('inf'), -10, 5, 20, 28, float('inf')]
labels = ["0", "1", "2", "3", "4"]

# 임시 DataFrame temp_df 정의
temp_df = origin_all.copy()

# direction을 숫자 레이블로 변환
temp_df['direction'] = temp_df['direction'].map(direction_map)

# 체감온도 단계 파생변수 추가
temp_df['feel_stage'] = pd.cut(temp_df['feel_temp'], bins=bins, labels=labels).astype(int)

In [13]:
# 필요한 칼럼만 추출
selected_cols = [
    'time',
    'station_number',
    'direction',
    'temp',
    'wind_speed',
    'rn_hr',
    'humidity',
    'solar',
    'congestion',
    'feel_stage'
]

# temp_df에서 필요한 컬럼만 train_data DataFrame에 저장
train_data = temp_df[selected_cols].copy()

# station_number, direction, time 순으로 정렬
train_data = train_data.sort_values(
    by=['station_number', 'direction', 'time']
).reset_index(drop=True)

# time 제거
train_data = train_data.iloc[:, 1:]

# 완성
train_data

,station_number,direction,temp,wind_speed,rn_hr,humidity,solar,congestion,feel_stage
0,150,0,-9.6,3.3,0.0,47.5,0.00,0,0
1,150,0,-9.7,2.0,0.0,47.5,0.00,0,1
2,150,0,-9.3,2.4,0.0,47.5,0.00,1,0
3,150,0,-9.3,1.7,0.0,47.5,0.00,2,0
4,150,0,-9.1,1.3,0.0,47.5,0.00,3,1
...,...,...,...,...,...,...,...,...,...
16369319,9006,1,1.4,0.0,0.5,92.1,0.03,15,1
16369320,9006,1,0.8,0.5,0.5,95.3,0.03,13,1
16369321,9006,1,0.4,0.1,0.5,96.2,0.03,14,1
16369322,9006,1,0.0,0.1,0.5,97.1,0.03,14,1


## 슬라이딩 윈도우

In [15]:
def extract_inputoutput_shift(df, lookback_time=2, predict_time=1):
    # 1) 과거 시점별 피처 생성
    feat_dfs = []
    for t in range(lookback_time):
        tmp = df.shift(t)
        feat_dfs.append(tmp)
    X = pd.concat(feat_dfs, axis=1)

    # 2) 예측 대상(타깃) 생성
    y = df[['congestion']].shift(-predict_time)

    # 3) NaN 제거 및 인덱스 리셋
    valid_idx = X.dropna().index.intersection(y.dropna().index)
    X = X.loc[valid_idx].reset_index(drop=True)
    y = y.loc[valid_idx].reset_index(drop=True)

    print("X, Y 데이터 분류 완료")
    return X, y

In [16]:
x, t = extract_inputoutput_shift(train_data)
x

X, Y 데이터 분류 완료


,station_number,direction,temp,wind_speed,rn_hr,humidity,solar,congestion,feel_stage,station_number,direction,temp,wind_speed,rn_hr,humidity,solar,congestion,feel_stage
0,150,0,-9.7,2.0,0.0,47.5,0.00,0,1,150.0,0.0,-9.6,3.3,0.0,47.5,0.00,0.0,0.0
1,150,0,-9.3,2.4,0.0,47.5,0.00,1,0,150.0,0.0,-9.7,2.0,0.0,47.5,0.00,0.0,1.0
2,150,0,-9.3,1.7,0.0,47.5,0.00,2,0,150.0,0.0,-9.3,2.4,0.0,47.5,0.00,1.0,0.0
3,150,0,-9.1,1.3,0.0,47.5,0.00,3,1,150.0,0.0,-9.3,1.7,0.0,47.5,0.00,2.0,0.0
4,150,0,-8.5,0.6,0.0,47.5,0.00,3,1,150.0,0.0,-9.1,1.3,0.0,47.5,0.00,3.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16369317,9006,1,2.6,0.3,0.5,86.6,0.03,19,1,9006.0,1.0,4.6,0.7,0.5,76.2,0.26,18.0,1.0
16369318,9006,1,1.4,0.0,0.5,92.1,0.03,15,1,9006.0,1.0,2.6,0.3,0.5,86.6,0.03,19.0,1.0
16369319,9006,1,0.8,0.5,0.5,95.3,0.03,13,1,9006.0,1.0,1.4,0.0,0.5,92.1,0.03,15.0,1.0
16369320,9006,1,0.4,0.1,0.5,96.2,0.03,14,1,9006.0,1.0,0.8,0.5,0.5,95.3,0.03,13.0,1.0


In [17]:
t

,congestion
0,1.0
1,2.0
2,3.0
3,3.0
4,5.0
...,...
16369317,15.0
16369318,13.0
16369319,14.0
16369320,14.0


# 데이터셋 나누기

In [19]:
x_train, x_test, t_train, t_test = train_test_split(x, t, test_size=0.2, shuffle=False)

print('x_train shape :', x_train.shape)
print('t_train shape :', t_train.shape)
print('x_test shape :', x_test.shape)
print('t_test shape :', t_test.shape)

x_train shape : (13095457, 18)
t_train shape : (13095457, 1)
x_test shape : (3273865, 18)
t_test shape : (3273865, 1)


In [20]:
timesteps = 2
feature = 9

x_train = np.array(x_train)
x_train = x_train.reshape(x_train.shape[0], timesteps, feature)

x_test = np.array(x_test)
x_test = x_test.reshape(x_test.shape[0], timesteps, feature)

t_train = np.array(t_train)
t_test = np.array(t_test)

print('reshape 후 x_train shape :', x_train.shape)
print('t_train shape :', t_train.shape)
print('reshape 후 x_test shape :', x_test.shape)
print('t_test shape :', t_test.shape)

reshape 후 x_train shape : (13095457, 2, 9)
t_train shape : (13095457, 1)
reshape 후 x_test shape : (3273865, 2, 9)
t_test shape : (3273865, 1)


In [21]:
cell_size = 128
timesteps = 2
feature = 9

model = Sequential(name='congestion_lstm_1')

model.add(LSTM(cell_size, input_shape=(timesteps, feature), return_sequences=True))
model.add(LSTM(cell_size))

model.add(Dense(1))

model.compile(loss=mse, optimizer=Adam(learning_rate = 0.001), metrics=[RootMeanSquaredError()])
model.summary()

/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "congestion_lstm_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 2, 128)         │        70,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 202,369 (790.50 KB)

 Trainable params: 202,369 (790.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(x_train, t_train, epochs = 20, batch_size = 32)

Epoch 1/20
125348/409234 ━━━━━━━━━━━━━━━━━━━━ 11:10 2ms/step - loss: 121.1202 - root_mean_squared_error: 10.9756

In [ ]:
pred = model.predict(x_test)

for i in range(1, 10):
    print('혼잡도 예측: ', round(pred[i][0]), '정답: ', round(t_test[i][0], 2))

In [ ]:
loss, rmse = model.evaluate(x_test, t_test, verbose=1)
print('test loss (MSE): ', round(loss, 6))
print('test RMSE: ', round(rmse, 2))